# Vazamento — o erro que não dá erro

**Capítulo I.3** do livro vivo [Ciência de Dados e Aprendizado de Máquina](https://machinelearning.ghdaru.com.br/i-3-dados.html).

Vazamento não levanta exceção. Ele **melhora** a métrica — e é por isso que passa. Este notebook mostra os três tipos que mais aparecem, e as divisões que os evitam.

Só biblioteca padrão.

In [ ]:
# --- roda igual na sua máquina e no Colab ------------------------------
# Na sua máquina: o notebook acha o repositório subindo de pasta.
# No Colab: não há repositório, então os arquivos necessários são baixados.
import pathlib, sys, urllib.request

RAW = "https://raw.githubusercontent.com/GHDaru/machinelearning/main/"
PRECISA = ['ml-zero/etapa-02/dados.py']

raiz = pathlib.Path.cwd()
for _ in range(5):
    if (raiz / "ml-zero").is_dir():
        break
    raiz = raiz.parent
else:
    raiz = pathlib.Path.cwd()

for rel in PRECISA:
    destino = raiz / rel
    if not destino.exists():
        destino.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(RAW + rel, destino)
        print("baixado:", rel)

sys.path.insert(0, str(raiz / "ml-zero/etapa-02"))
RAIZ = raiz
print("pronto.")

## 1. A coluna que sabe demais

Um atributo que, sozinho, prevê o rótulo quase perfeitamente costuma ser consequência dele, não causa.

In [ ]:
import random
from dados import detectar_vazamento_obvio

# 200 linhas. Com poucas linhas QUALQUER coluna separa perfeitamente por acaso —
# e o detector acusaria todas. Vazamento é um fenômeno estatístico: precisa de
# dado suficiente para ser distinguível de coincidência.
random.seed(7)
n = 200
y = [1 if random.random() < 0.35 else 0 for _ in range(n)]

colunas = {
    "idade":           [random.randint(18, 70) for _ in range(n)],
    "regiao":          [random.choice("ABC") for _ in range(n)],
    "valor_do_seguro": [900 if alvo else 0 for alvo in y],   # <- consequência do rótulo
}

suspeitas = detectar_vazamento_obvio(colunas, y)
for s in suspeitas:
    print(f"{s.coluna:18s} {s.motivo}")
print()
print("colunas acusadas:", [s.coluna for s in suspeitas] or "nenhuma")

## 2. A divisão que respeita grupos

Se o mesmo cliente aparece em treino e em teste, o modelo decora a pessoa, não o padrão.

In [ ]:
from dados import dividir_por_grupo, vazou_entre

grupos = ["ana", "ana", "bruno", "bruno", "carla", "carla", "davi", "davi", "eva", "eva"]
tr, va, te = dividir_por_grupo(grupos)

print("treino   ", sorted({grupos[i] for i in tr}))
print("validação", sorted({grupos[i] for i in va}))
print("teste    ", sorted({grupos[i] for i in te}))
print("grupos vazados:", vazou_entre(grupos, tr, va, te) or "nenhum")

## 3. A divisão que respeita o tempo

Se o problema tem tempo, treinar no futuro para prever o passado infla a métrica — e nada avisa.

In [ ]:
from dados import dividir_por_tempo

tempos = [f"2026-01-{d:02d}" for d in range(1, 11)]
tr, va, te = dividir_por_tempo(tempos)

print("treino   ", [tempos[i] for i in tr])
print("validação", [tempos[i] for i in va])
print("teste    ", [tempos[i] for i in te])

## O que levar

- Vazamento **melhora** a métrica. Ele nunca se apresenta como erro.
- Divisão aleatória é o padrão errado quando há **grupo** ou **tempo** no problema.
- Suspeite de todo atributo bom demais: pergunte *"eu teria esse valor no momento da previsão?"*

Continue em [02 — Qualidade e Vazamento](https://machinelearning.ghdaru.com.br/i-3-dados.html).